In [1]:
import os
import glob
import re
from nbconvert import HTMLExporter
from traitlets.config import Config
import nbformat

def update_ipynb_links_to_html(html_body):
    """
    Post-process HTML body to replace .ipynb links with .html equivalents, preserving fragments (#anchors).
    
    Args:
        html_body (str): The HTML content as a string.
    
    Returns:
        str: Updated HTML body.
    """
    # Regex to match href=".../file.ipynb#fragment" or href="file.ipynb" (double quotes only)
    # Captures: group(1) = path before .ipynb, group(2) = after .ipynb (e.g., #subheading or query)
    pattern = r'(href=")([^"]*\.)?(?P<before>[^"#\.]*)ipynb(?P<after>#?[^"]*)"'
    def replacer(match):
        # Replace .ipynb with .html, preserving path, before, and after (including #anchor)
        return f'{match.group(1)}{match.group(2)}{match.group("before")}html{match.group("after")}"'
    
    updated_body = re.sub(pattern, replacer, html_body)
    return updated_body

def convert_ipynb_to_html_with_link_updates(directory_path):
    """
    Recursively convert all .ipynb files to HTML, preserving relative images and updating .ipynb links (with anchors) to .html.
    Special handling: Rename "OPEN ME FIRST - README - Main Menu.ipynb" output to root index.html.
    
    Args:
        directory_path (str): Path to the root directory to search for .ipynb files.
    """
    # Find all .ipynb files recursively (sorted for consistent order)
    ipynb_files = sorted(glob.glob(os.path.join(directory_path, '**', '*.ipynb'), recursive=True))
    
    if not ipynb_files:
        print("No .ipynb files found in the directory.")
        return
    
    # Configure exporter for relative paths (no embedding)
    c = Config()
    c.HTMLExporter.embed_images = False
    exporter = HTMLExporter(config=c)
    
    root_dir = os.path.abspath(directory_path)
    target_notebook_name = "OPEN ME FIRST - README - Main Menu.ipynb"  # Exact filename with spaces
    
    for ipynb_path in ipynb_files:
        try:
            # Read the notebook
            with open(ipynb_path, 'r', encoding='utf-8') as f:
                notebook = nbformat.read(f, as_version=4)
            
            # Convert to HTML
            body, resources = exporter.from_notebook_node(notebook)
            
            # Post-process: Update .ipynb links to .html, including anchors
            body = update_ipynb_links_to_html(body)
            
            # Determine output path
            notebook_name = os.path.basename(ipynb_path)
            if notebook_name == target_notebook_name:
                # Special case: Rename to root index.html
                html_filename = 'index.html'
                html_path = os.path.join(root_dir, html_filename)
                resource_dir_name = 'index_files'  # For attachments
                print(f"Special conversion: Renaming to root {html_path}")
            else:
                # Standard: .html in same dir
                html_dir = os.path.dirname(ipynb_path)
                html_filename = os.path.splitext(notebook_name)[0] + '.html'
                html_path = os.path.join(html_dir, html_filename)
                resource_dir_name = f"{os.path.splitext(html_filename)[0]}_files"
            
            # Write updated HTML file
            with open(html_path, 'w', encoding='utf-8') as f:
                f.write(body)
            
            # Extract attachments to _files if present
            if resources and 'files' in resources:
                if notebook_name == target_notebook_name:
                    resource_dir = os.path.join(root_dir, resource_dir_name)
                else:
                    resource_dir = os.path.join(os.path.dirname(html_path), resource_dir_name)
                os.makedirs(resource_dir, exist_ok=True)
                for file_path, content in resources.get('files', {}).items():
                    with open(os.path.join(resource_dir, file_path), 'wb') as res_file:
                        res_file.write(content)
                print(f"Extracted attachments to: {resource_dir}")
            
            print(f"Converted and updated links (with anchors): {ipynb_path} -> {html_path}")
            
        except Exception as e:
            print(f"Error converting {ipynb_path}: {str(e)}")

# Example usage: Replace '.' with your directory path
directory = '.'  # Current directory; change as needed
convert_ipynb_to_html_with_link_updates(directory)

Converted and updated links (with anchors): .\CAD_Files\3 Port Reservoir\Manufacturing Notes.ipynb -> .\CAD_Files\3 Port Reservoir\Manufacturing Notes.html
Converted and updated links (with anchors): .\ChronoSeq_Overview.ipynb -> .\ChronoSeq_Overview.html
Special conversion: Renaming to root C:\Users\ChronoSeq\ChronoSeq\index.html
Converted and updated links (with anchors): .\OPEN ME FIRST - README - Main Menu.ipynb -> C:\Users\ChronoSeq\ChronoSeq\index.html
Converted and updated links (with anchors): .\convertToHTML.ipynb -> .\convertToHTML.html
Converted and updated links (with anchors): .\instructions_for_assembling_sensirion_flow_sensor.ipynb -> .\instructions_for_assembling_sensirion_flow_sensor.html
Converted and updated links (with anchors): .\instructions_for_assembling_valve_controller.ipynb -> .\instructions_for_assembling_valve_controller.html
Converted and updated links (with anchors): .\instructions_for_assembling_valve_controllerV2.ipynb -> .\instructions_for_assembling_v

C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted and updated links (with anchors): .\instructions_for_assembling_valve_controller_ESP32.ipynb -> .\instructions_for_assembling_valve_controller_ESP32.html
Converted and updated links (with anchors): .\instructions_for_assembling_vortex_relay_controller.ipynb -> .\instructions_for_assembling_vortex_relay_controller.html


C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted and updated links (with anchors): .\instructions_for_assembling_xy_robot_and_ice_boxes.ipynb -> .\instructions_for_assembling_xy_robot_and_ice_boxes.html
Converted and updated links (with anchors): .\instructions_for_assembling_xyz_robot.ipynb -> .\instructions_for_assembling_xyz_robot.html
Converted and updated links (with anchors): .\instructions_for_device_assembly_and_setup.ipynb -> .\instructions_for_device_assembly_and_setup.html
Converted and updated links (with anchors): .\instructions_for_setting_coordinates_for_xyz_robot.ipynb -> .\instructions_for_setting_coordinates_for_xyz_robot.html
Converted and updated links (with anchors): .\instructions_for_setting_up_valves_tubing_and_reservoirs.ipynb -> .\instructions_for_setting_up_valves_tubing_and_reservoirs.html
Converted and updated links (with anchors): .\protocol_and_software_for_running_chronoseq_device-OriginalDevice.ipynb -> .\protocol_and_software_for_running_chronoseq_device-OriginalDevice.html
Converted and up

...

Converted and updated links (with anchors): .\protocol_for_single_cell_scale_up_chronoseqv4_dropseq_bead_modification.ipynb -> .\protocol_for_single_cell_scale_up_chronoseqv4_dropseq_bead_modification.html


C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted and updated links (with anchors): .\protocol_for_single_cell_scale_up_chronoseqv5_dropseq_bead_modification.ipynb -> .\protocol_for_single_cell_scale_up_chronoseqv5_dropseq_bead_modification.html
Converted and updated links (with anchors): .\protocol_for_tagmentation_with_KAPA_PCR.ipynb -> .\protocol_for_tagmentation_with_KAPA_PCR.html
Converted and updated links (with anchors): .\protocol_library_preparation_for_dropseq_chronoseq_beads.ipynb -> .\protocol_library_preparation_for_dropseq_chronoseq_beads.html
Converted and updated links (with anchors): .\protocol_library_preparation_for_dropseq_chronoseq_beads_previous_version.ipynb -> .\protocol_library_preparation_for_dropseq_chronoseq_beads_previous_version.html
Converted and updated links (with anchors): .\qPCR_validation_files\K562_Costimulation_2_Samples\qPCR_analysis_publication_quality_plots.ipynb -> .\qPCR_validation_files\K562_Costimulation_2_Samples\qPCR_analysis_publication_quality_plots.html


C:\Users\Kanishk Asthana\.conda\envs\chrono\lib\site-packages\nbformat\__init__.py:92: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Converted and updated links (with anchors): .\qPCR_validation_files\qPCR Analysis.ipynb -> .\qPCR_validation_files\qPCR Analysis.html
Converted and updated links (with anchors): .\removeLinksFromNBConvertHTML.ipynb -> .\removeLinksFromNBConvertHTML.html
